In [ ]:
"""
MBA Strategic Analysis - 3 Executive Graphs
Fixed syntax error - displays interactive graph in browser
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

print("🎓 MBA STRATEGIC ANALYSIS - EXECUTIVE DASHBOARD")
print("="*55)

# =====================================
# DATA PREPARATION
# =====================================

def create_sample_data():
    """Create comprehensive sample dataset"""
    print("🔄 Creating sample data...")
    np.random.seed(42)
    n_stores = 45
    n_weeks = 52
    dates = pd.date_range('2021-01-01', periods=n_weeks, freq='W')
    
    data = []
    for store in range(1, n_stores + 1):
        store_type = np.random.choice(['A', 'B', 'C'], p=[0.3, 0.4, 0.3])
        store_size = np.random.randint(50000, 200000)
        
        for date in dates:
            base_sales = np.random.normal(50000, 15000)
            seasonal_factor = 1 + 0.3 * np.sin(2 * np.pi * date.dayofyear / 365)
            weekly_sales = max(0, base_sales * seasonal_factor)
            
            if date.month == 12: weekly_sales *= 1.4
            elif date.month == 6: weekly_sales *= 0.8
                
            data.append({
                'Store': store,
                'Date': date,
                'Weekly_Sales': weekly_sales,
                'Type': store_type,
                'Size': store_size,
                'Marketing': np.random.uniform(500, 3000)
            })
    
    df = pd.DataFrame(data)
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    print(f"✅ Sample data created: {df.shape[0]:,} records")
    return df

# =====================================
# GRAPH 1: STORE PERFORMANCE (Static)
# =====================================

def create_store_performance_chart(df):
    """Create static store performance visualization"""
    print("\n📈 GRAPH 1: Store Performance Analysis")
    
    # Aggregate store metrics
    store_metrics = df.groupby('Store').agg({
        'Weekly_Sales': 'sum',
        'Type': 'first',
        'Size': 'first',
        'Marketing': 'sum'
    })
    store_metrics['ROI'] = store_metrics['Weekly_Sales'] / store_metrics['Marketing']
    
    # Create figure
    plt.figure(figsize=(10, 6))
    
    # Color mapping
    colors = {'A': '#FF6B6B', 'B': '#4ECDC4', 'C': '#45B7D1'}
    
    # Bubble chart
    for store_type, group in store_metrics.groupby('Type'):
        plt.scatter(
            group['Size'] / 1000,
            group['Weekly_Sales'] / 1e6,
            s=group['ROI'] * 50,
            c=colors[store_type],
            alpha=0.7,
            label=f'Type {store_type}',
            edgecolor='w',
            linewidth=0.5
        )
    
    # Add labels and title
    plt.title('Store Performance: Size vs Sales vs ROI', fontsize=14)
    plt.xlabel('Store Size (Thousands of Sq Ft)', fontsize=10)
    plt.ylabel('Total Sales ($ Millions)', fontsize=10)
    plt.grid(alpha=0.1)
    plt.legend(title='Store Type', loc='best')
    
    # Add ROI reference
    plt.figtext(0.5, 0.01, "Bubble size represents Marketing ROI", 
                ha="center", fontsize=9,
                bbox=dict(facecolor='white', alpha=0.5))
    
    # Save and return
    plt.tight_layout()
    plt.savefig('store_performance.png', dpi=100)
    print("✅ Saved store_performance.png")
    
    return store_metrics

# =====================================
# GRAPH 2: SEASONAL TRENDS (Static)
# =====================================

def create_seasonal_chart(df):
    """Create static seasonal sales chart"""
    print("\n📈 GRAPH 2: Seasonal Sales Analysis")
    
    # Prepare data
    monthly_sales = df.groupby(['Year', 'Month'])['Weekly_Sales'].sum().reset_index()
    monthly_avg = monthly_sales.groupby('Month')['Weekly_Sales'].mean().reset_index()
    
    # Create figure
    plt.figure(figsize=(10, 5))
    
    # Plot each year
    years = monthly_sales['Year'].unique()
    for year in years:
        year_data = monthly_sales[monthly_sales['Year'] == year]
        plt.plot(year_data['Month'], year_data['Weekly_Sales'] / 1e6, 
                label=str(year), linewidth=1.5)
    
    # Plot average
    plt.plot(monthly_avg['Month'], monthly_avg['Weekly_Sales'] / 1e6, 
            'k--', linewidth=2, label='Average')
    
    # Add labels and title
    plt.title('Monthly Sales Trends', fontsize=14)
    plt.xlabel('Month', fontsize=10)
    plt.ylabel('Sales ($ Millions)', fontsize=10)
    plt.xticks(range(1, 13), ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
    plt.grid(alpha=0.1)
    plt.legend(title='Year', fontsize=8)
    
    # Highlight peak month
    peak_month = monthly_avg.loc[monthly_avg['Weekly_Sales'].idxmax(), 'Month']
    plt.axvline(x=peak_month, color='r', linestyle=':', alpha=0.7)
    plt.text(peak_month, plt.ylim()[1]*0.9, 'Peak', color='r', fontsize=9, ha='center')
    
    # Save and return
    plt.tight_layout()
    plt.savefig('seasonal_trends.png', dpi=100)
    print("✅ Saved seasonal_trends.png")
    
    return monthly_sales

# =====================================
# GRAPH 3: INTERACTIVE DASHBOARD
# =====================================

def create_interactive_dashboard(df):
    """Create interactive dashboard that opens automatically"""
    print("\n📈 GRAPH 3: Interactive Performance Dashboard")
    
    # Prepare data
    store_summary = df.groupby(['Store', 'Type']).agg({
        'Weekly_Sales': 'sum',
        'Marketing': 'sum',
        'Size': 'first'
    }).reset_index()
    store_summary['ROI'] = store_summary['Weekly_Sales'] / store_summary['Marketing']
    
    # Create figure
    fig = go.Figure()
    
    # Color mapping
    colors = {'A': '#FF6B6B', 'B': '#4ECDC4', 'C': '#45B7D1'}
    
    # Add traces for each store type
    for store_type in ['A', 'B', 'C']:
        store_data = store_summary[store_summary['Type'] == store_type]
        
        # Create hover text
        hover_text = []
        for _, row in store_data.iterrows():
            text = (f"<b>Store {row['Store']}</b><br>"
                    f"Type: {row['Type']}<br>"
                    f"Size: {row['Size']/1000:.1f}K sq ft<br>"
                    f"Sales: ${row['Weekly_Sales']/1e6:.2f}M<br>"
                    f"Marketing: ${row['Marketing']/1e3:.1f}K<br>"
                    f"ROI: {row['ROI']:.1f}x")
            hover_text.append(text)
        
        fig.add_trace(go.Scatter(
            x=store_data['Marketing'] / 1e3,
            y=store_data['Weekly_Sales'] / 1e6,
            mode='markers',
            name=f'Type {store_type}',
            marker=dict(
                size=store_data['Size'] / 5000,
                color=colors[store_type],
                opacity=0.7,
                line=dict(width=0.5, color='DarkSlateGrey')
            ),
            text=hover_text,
            hovertemplate='%{text}<extra></extra>'
        ))
    
    # Update layout
    fig.update_layout(
        title='Marketing vs Sales Performance',
        title_x=0.5,
        title_font_size=18,
        xaxis_title='Marketing Investment ($ Thousands)',
        yaxis_title='Total Sales ($ Millions)',
        template='plotly_white',
        hoverlabel=dict(bgcolor="white", font_size=10),
        legend_title_text='Store Type',
        height=500,
        width=800
    )
    
    # Add annotations
    fig.add_annotation(
        x=0.05, y=0.95,
        xref="paper", yref="paper",
        text="Bubble size = Store size",
        showarrow=False,
        font=dict(size=10, color="gray"),
        bgcolor="white"
    )
    
    # Show the interactive plot
    print("🖥️ Opening interactive dashboard in your browser...")
    fig.show()
    
    return store_summary

# =====================================
# EXECUTIVE SUMMARY
# =====================================

def generate_executive_summary(store_metrics, seasonal_data):
    """Generate executive summary with key insights"""
    print("\n📊 EXECUTIVE SUMMARY")
    print("="*50)
    
    # Key metrics
    total_sales = store_metrics['Weekly_Sales'].sum() / 1e6
    best_store = store_metrics.loc[store_metrics['Weekly_Sales'].idxmax()]
    best_month = seasonal_data.groupby('Month')['Weekly_Sales'].mean().idxmax()
    month_names = ['January', 'February', 'March', 'April', 'May', 'June', 
                  'July', 'August', 'September', 'October', 'November', 'December']
    month_name = month_names[best_month-1]
    
    print(f"💰 Total Sales: ${total_sales:.2f}M")
    print(f"🏆 Top Store: #{int(best_store.name)} (Type {best_store['Type']})")
    print(f"   - Sales: ${best_store['Weekly_Sales']/1e6:.2f}M")
    print(f"   - Size: {best_store['Size']/1000:.1f}K sq ft")
    print(f"   - ROI: {best_store['ROI']:.1f}x")
    print(f"📅 Peak Sales Month: {month_name}")
    
    # Recommendations
    print("\n📋 RECOMMENDATIONS:")
    print(f"1. Focus on Type {best_store['Type']} stores (top performer)")
    print(f"2. Analyze Store #{int(best_store.name)} best practices")
    print(f"3. Increase marketing during {month_name}")
    print("4. Optimize store layouts for efficiency")

# =====================================
# MAIN EXECUTION
# =====================================

def main():
    """Main function to run the analysis"""
    # Create sample data
    df = create_sample_data()
    
    # Create and show interactive dashboard first
    print("\n🚀 Launching interactive dashboard...")
    store_metrics = create_interactive_dashboard(df)
    
    # Create static plots
    print("\n📊 Generating static plots...")
    store_metrics = create_store_performance_chart(df)
    seasonal_data = create_seasonal_chart(df)
    
    # Generate summary
    generate_executive_summary(store_metrics, seasonal_data)
    
    # Show static plots at the end
    print("\n🖼️ Displaying static plots...")
    plt.show()
    
    print("\n✅ Analysis Complete!")
    print("💾 Files saved:")
    print("   - store_performance.png")
    print("   - seasonal_trends.png")

if __name__ == "__main__":
    main()
    